<a href="https://colab.research.google.com/github/Yumna-Zafar/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yumna-Zafar/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yumna-Zafar/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir("..")
    if os.path.basename(os.getcwd()) == "work":
        os.chdir("..")

print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship/flyrank-ml-internship


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane: Refresh / Content Opportunity Scoring.

Task type: classification, used to produce a ranked score. The core prediction is binary
(will this page be flagged as declining, yes/no), but the output isn't consumed as a
label — it's consumed as a probability that ranks pages from "review first" to "review
last." So the task type is classification-for-ranking / scoring: a model that outputs
a probability, which becomes the sort key for a review queue.

Why this type: the real decision here isn't "declining or not" in isolation — it's
"given limited reviewer capacity, which pages should a human look at first?" That's a
ranking/prioritization problem wearing a classification model's clothes.

In [9]:
import pandas as pd, numpy as np
print("Lane: Refresh / Content Opportunity Scoring")
print("Task type: binary classification -> probability used as a ranking score")


Lane: Refresh / Content Opportunity Scoring
Task type: binary classification -> probability used as a ranking score


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target/proxy: is_declining_label = (trend_direction == "down")

This is a proxy, not a true future outcome. It's a bucket calculated from the CURRENT
window (the page's most recent trend direction), not something that happens after a
defined decision point. The docs are explicit that this is a "beginner proxy label" —
useful to prove the workflow, but weaker than a real target.

A stronger, future-looking version would be: using features from the prior 90 days,
predict decline over the NEXT 30 days. That avoids the proxy's core weakness (it can't
leak into itself, because the label window is strictly after the feature window). I'm
using the simpler current-window proxy here since this is the framing exercise, not
the final model — but I'm naming its weakness honestly rather than hiding it.

In [10]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Declining rate (proxy label):", round(df["is_declining_label"].mean(), 3))


Declining rate (proxy label): 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@50.

Why this one: the real decision this supports is "a reviewer has capacity to check
roughly 50 pages this cycle — which 50 should they be?" Precision@50 answers exactly
that: of the top 50 pages the model flags, what fraction are actually in the declining
proxy group? A high number means the reviewer's limited time is well spent; a low
number means they'd be chasing false alarms.

I'm not using plain accuracy, because accuracy treats all 30,000 rows as equally
important to get right — but nobody reviews 30,000 pages. Precision@K matches how the
output is actually used: as a short, ordered list, not a blanket classification.

In [11]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Metric: Precision@50 -- fraction of the top 50 ranked pages that are truly declining")


Metric: Precision@50 -- fraction of the top 50 ranked pages that are truly declining


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content page (content_id), from a single client, at
one point in time (the most recent snapshot in the starter dataset).

This is NOT one row per day, per query, or per client -- it's page-level, so every
column describes a single page's current state (its impressions, position, freshness,
word count, and so on).

In [12]:
unit_view = df[["content_id", "client_id", "impressions_90d", "avg_position",
                 "days_since_last_update", "word_count", "trend_direction",
                 "is_declining_label"]].head(10)
print("Shape:", df.shape, " -> one row per content_id")
unit_view

Shape: (30000, 45)  -> one row per content_id


,content_id,client_id,impressions_90d,avg_position,days_since_last_update,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,20,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,25,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,20,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,22,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,14,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,20,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,7.0,20,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,21.2,22,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,46.0,20,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,4.9,104,NaN,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like the hand-written stale x visible rule from Week 2) can only combine
a small, fixed number of thresholds a human picks in advance. Real refresh-worthiness
depends on several signals interacting in ways that aren't obvious from a single
if-statement: a page can be stale AND high-traffic but stable, or fresh but declining
fast, or low-position but rapidly growing. A hand rule has to pick 2-3 factors and a
fixed cutoff for each; a model can weigh six or more signals at once and find where
the real cutoffs actually sit in the data, not where I guessed they'd sit.

The starter pipeline's own numbers back this up (from docs/ml-intern-dataset-and-lane-guide.md,
outputs/model_report.md): the hand-tuned baseline rule scores Precision@50 = 0.240,
while a random forest on the same data scores 0.740 -- roughly 3x more of the top 50
flagged pages are genuinely declining. That gap is the evidence: the useful pattern
here is too tangled across multiple correlated signals for one fixed rule to capture,
but a model can still explain itself afterward (feature importance, tree paths), so
we don't lose interpretability to get the lift.

In [13]:
print("Baseline rule Precision@50 (from outputs/model_report.md): 0.240")
print("Random forest Precision@50 (from outputs/model_report.md): 0.740")
print("Observed lift: roughly 3x more true positives in the reviewer's top 50")

Baseline rule Precision@50 (from outputs/model_report.md): 0.240
Random forest Precision@50 (from outputs/model_report.md): 0.740
Observed lift: roughly 3x more true positives in the reviewer's top 50


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.